
<h1 id="Tutorial:-Trace-Direct-vs-Programmatic-Tool-Calling-%E2%80%94-Inventory-Replenishment">Tutorial: Trace Direct vs Programmatic Tool Calling — Inventory Replenishment<a class="anchor-link" href="#Tutorial:-Trace-Direct-vs-Programmatic-Tool-Calling-%E2%80%94-Inventory-Replenishment">¶</a></h1><p><strong>Audience:</strong> Engineers evaluating OpenAI Responses API tool orchestration.</p>
<p><strong>Prerequisites:</strong> Python, the Responses API tool-calling loop, and the difference between a model turn and a host-side tool execution.</p>
<p><strong>Learning goals:</strong> By the end, you can inspect one comparison trace containing Direct and Programmatic sibling arms, follow observable tool-calling events, and compare quality, host round trips, payload size, tokens, latency, and estimated cost.</p>
<blockquote>
<p>Scope: traces expose observable API and tool lifecycle events. They do not expose hidden model reasoning.</p>
</blockquote>



<h2 id="Outline">Outline<a class="anchor-link" href="#Outline">¶</a></h2><ol>
<li>Configure safe live and Trace export controls.</li>
<li>Build the deterministic inventory fixture.</li>
<li>Inspect the shared tools and arm-specific orchestration.</li>
<li>Run one optional Direct/PTC comparison inside a parent OpenAI Trace.</li>
<li>Compare the semantic event timeline, quality, and resource metrics.</li>
<li>Practice interpreting host round trips and intermediate payloads.</li>
</ol>



<h2 id="1.-Setup">1. Setup<a class="anchor-link" href="#1.-Setup">¶</a></h2><p>The Trace adapter wraps the existing raw Responses API client and deterministic scenario. It records semantic events without replacing the benchmark's orchestration loop.</p>


In [ ]:
from __future__ import annotations

import importlib.util
import os
import sys
import uuid
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ptc_benchmark.config import configured_model, load_local_environment, require_api_key
from ptc_benchmark.inventory import build_inventory_dataset
from ptc_benchmark.inventory_evaluation import evaluate_inventory_run
from ptc_benchmark.pricing import estimate_run_cost, load_pricing_catalog
from ptc_benchmark.reporting import markdown_table
from ptc_benchmark.runner import RunConfig
from ptc_trace_demo.inventory_trace import (
    configure_trace_error_logging,
    run_inventory_trace_comparison,
)



<h3 id="Safe-execution-controls">Safe execution controls<a class="anchor-link" href="#Safe-execution-controls">¶</a></h3><p>Live API usage, OpenAI Trace export, and detailed trace-error logging use separate opt-ins. Full synthetic payloads remain local for timeline inspection; the OpenAI Trace backend receives bounded summaries only. Set <code>SHOW_TRACE_ERROR_DETAILS=True</code> temporarily to include the Trace ingest response body in errors. Detailed logs can contain sensitive model or tool data, so keep it <code>False</code> for normal use.</p>


In [ ]:
RUN_LIVE = True # Set to True to enable API calls and associated cost.
EXPORT_OPENAI_TRACE = True # Set to True to export bounded summaries; requires RUN_LIVE = True.
SHOW_TRACE_ERROR_DETAILS = True # Set to True temporarily to show Trace ingest error bodies.
INCLUDE_LOCAL_PAYLOADS = True  # Retain full deterministic payloads in the local timeline only.

configure_trace_error_logging(show_details=SHOW_TRACE_ERROR_DETAILS)

SCALE = "small"  # small=3 SKUs, medium=10, large=30
MODEL = configured_model("gpt-5.6")
REASONING_EFFORT = "medium"
MAX_REQUESTS = 16  # Bounded continuation budget for both comparison arms.

PRICING_PATH = Path(
    os.getenv(
        "OPENAI_PRICING_PATH",
        PROJECT_ROOT / "pricing" / "openai_pricing_2026-08-06.json",
    )
)

print({
    "RUN_LIVE": RUN_LIVE,
    "EXPORT_OPENAI_TRACE": EXPORT_OPENAI_TRACE,
    "show_trace_error_details": SHOW_TRACE_ERROR_DETAILS,
    "include_local_payloads": INCLUDE_LOCAL_PAYLOADS,
    "scale": SCALE,
    "model": MODEL,
    "reasoning_effort": REASONING_EFFORT,
    "max_requests": MAX_REQUESTS,
})



<h2 id="2.-Deterministic-fixture-and-quality-oracle">2. Deterministic fixture and quality oracle<a class="anchor-link" href="#2.-Deterministic-fixture-and-quality-oracle">¶</a></h2><p>Both arms receive the same SKUs, tool schemas, model settings, and expected replenishment plan. Quality remains a hard gate before cost is interpreted.</p>


In [ ]:
dataset = build_inventory_dataset(SCALE)
oracle = dataset.expected_plan()

print({
    "case_id": dataset.case_id,
    "sku_count": len(dataset.skus),
    "expected_tool_calls_per_arm": len(dataset.skus) * 3,
    "expected_recommendations": len(oracle["recommendations"]),
})



<h2 id="3.-What-the-Trace-should-reveal">3. What the Trace should reveal<a class="anchor-link" href="#3.-What-the-Trace-should-reveal">¶</a></h2><div class="highlight"><pre><span></span>Inventory comparison trace
├── direct_arm
│   ├── model_request → function_call observations
│   ├── host tool executions and outputs
│   └── next model_request → final assistant message
└── programmatic_arm
    ├── model_request → program + function_call observations
    ├── hosted-program-linked tool executions and outputs
    └── next model_request → program_output + final assistant message
</pre></div>
<p>A <code>caller_id</code> on PTC function calls links each call to its generated program. Direct calls should have no program caller.</p>


In [ ]:
contract_rows = []
prompt_sections = []
for arm in ("direct", "programmatic"):
    instructions, user_input = dataset.prompt(arm)
    tools = dataset.tool_definitions(arm)
    contract_rows.append({
        "arm": arm,
        "function_tools": sum(tool["type"] == "function" for tool in tools),
        "ptc_tool_enabled": any(tool["type"] == "programmatic_tool_calling" for tool in tools),
        "allowed_callers": sorted({
            caller
            for tool in tools
            if tool["type"] == "function"
            for caller in tool.get("allowed_callers", [])
        }),
        "user_prompt_chars": len(user_input),
    })
    prompt_sections.append(
        f"### {arm.title()}\n\n"
        f"**Instructions**\n\n```text\n{instructions}\n```\n\n"
        f"**User Prompt**\n\n```text\n{user_input}\n```"
    )

display(Markdown(markdown_table(contract_rows)))
display(Markdown("\n\n".join(prompt_sections)))



<h3 id="Output-example-for-Section-3:-What-the-Trace-should-reveal">Output example for Section 3: What the Trace should reveal<a class="anchor-link" href="#Output-example-for-Section-3:-What-the-Trace-should-reveal">¶</a></h3><p>The following Markdown preserves an example of the output produced in this section.</p>
<table>
<thead>
<tr>
<th>arm</th>
<th>function_tools</th>
<th>ptc_tool_enabled</th>
<th>allowed_callers</th>
<th>user_prompt_chars</th>
</tr>
</thead>
<tbody>
<tr>
<td>direct</td>
<td>3</td>
<td>False</td>
<td>['direct']</td>
<td>67</td>
</tr>
<tr>
<td>programmatic</td>
<td>3</td>
<td>True</td>
<td>['programmatic']</td>
<td>67</td>
</tr>
</tbody>
</table>
<h3 id="Direct">Direct<a class="anchor-link" href="#Direct">¶</a></h3><p><strong>Instructions</strong></p>
<div class="highlight"><pre><span></span>&lt;task_contract&gt;
You are preparing a deterministic inventory replenishment plan as of 2026-08-10.

For every SKU in this exact list, call get_inventory, get_weekly_demand, and
get_inbound_shipments exactly once: sku-001, sku-002, sku-003.

For each SKU:
1. available_units = sum(on_hand_units - reserved_units) across warehouses.
2. forecast_units = sum(units) across all seven daily_forecast rows.
3. inbound_units = sum(units) only for shipments with status "scheduled" and
   eta_date on or before 2026-08-17.
4. reorder_units = max(forecast_units + 5 - available_units - inbound_units, 0).

Keep only positive reorder quantities. Sort by reorder_units descending and then
sku ascending. The structured result must be exactly one JSON object with this shape:
{"recommendations":[{"sku":"...","available_units":0,"forecast_units":0,
"inbound_units":0,"reorder_units":0}],"total_reorder_units":0}

The final assistant message must contain:
RESULT_JSON: &lt;the exact one-line JSON object&gt;
EXPLANATION: &lt;a concise explanation that cites every recommended SKU and all four
source/calculated unit values for that SKU&gt;.
Do not invent values and do not omit evidence from the explanation.
&lt;/task_contract&gt;

&lt;tool_orchestration&gt;
Use Direct Tool Calling. Call the functions directly and issue independent calls in
parallel when possible. Use the returned tool data to calculate the result. Do not
write or execute a programmatic_tool_calling program.
&lt;/tool_orchestration&gt;
</pre></div>
<p><strong>User Prompt</strong></p>
<div class="highlight"><pre><span></span>Which products should we reorder this week, and in what quantities?
</pre></div>
<h3 id="Programmatic">Programmatic<a class="anchor-link" href="#Programmatic">¶</a></h3><p><strong>Instructions</strong></p>
<div class="highlight"><pre><span></span>&lt;task_contract&gt;
You are preparing a deterministic inventory replenishment plan as of 2026-08-10.

For every SKU in this exact list, call get_inventory, get_weekly_demand, and
get_inbound_shipments exactly once: sku-001, sku-002, sku-003.

For each SKU:
1. available_units = sum(on_hand_units - reserved_units) across warehouses.
2. forecast_units = sum(units) across all seven daily_forecast rows.
3. inbound_units = sum(units) only for shipments with status "scheduled" and
   eta_date on or before 2026-08-17.
4. reorder_units = max(forecast_units + 5 - available_units - inbound_units, 0).

Keep only positive reorder quantities. Sort by reorder_units descending and then
sku ascending. The structured result must be exactly one JSON object with this shape:
{"recommendations":[{"sku":"...","available_units":0,"forecast_units":0,
"inbound_units":0,"reorder_units":0}],"total_reorder_units":0}

The final assistant message must contain:
RESULT_JSON: &lt;the exact one-line JSON object&gt;
EXPLANATION: &lt;a concise explanation that cites every recommended SKU and all four
source/calculated unit values for that SKU&gt;.
Do not invent values and do not omit evidence from the explanation.
&lt;/task_contract&gt;

&lt;tool_orchestration&gt;
Use Programmatic Tool Calling for the complete lookup and calculation stage. Create
all tool-call promises before awaiting them and resolve them with Promise.all. Perform
all filtering, summation, sorting, and reduction inside the generated JavaScript.
Emit exactly the required JSON object from the program with text(JSON.stringify(result)).
After the program completes, write the required RESULT_JSON and EXPLANATION final message.
Do not call the inventory functions directly.
&lt;/tool_orchestration&gt;
</pre></div>
<p><strong>User Prompt</strong></p>
<div class="highlight"><pre><span></span>Which products should we reorder this week, and in what quantities?
</pre></div>



<h2 id="4.-Optional-live-comparison-and-OpenAI-Trace-export">4. Optional live comparison and OpenAI Trace export<a class="anchor-link" href="#4.-Optional-live-comparison-and-OpenAI-Trace-export">¶</a></h2><p>Set <code>RUN_LIVE=True</code> for API execution. Also set <code>EXPORT_OPENAI_TRACE=True</code> to send bounded event summaries to the Trace backend. Full payloads stay in the local comparison object. Install the optional exporter with <code>uv sync --extra dev --extra trace</code>.</p>


In [ ]:
trace_comparison = None
trace_metric_rows = []

if not RUN_LIVE:
    print("Live comparison skipped. Set RUN_LIVE = True to opt in to API usage and cost.")
elif EXPORT_OPENAI_TRACE and importlib.util.find_spec("agents") is None:
    raise RuntimeError(
        "Trace export requires the optional dependency: uv sync --extra dev --extra trace"
    )
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    comparison_id = f"inventory-trace-{uuid.uuid4().hex[:10]}"
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=MAX_REQUESTS,
    )
    trace_comparison = run_inventory_trace_comparison(
        client=OpenAI(),
        dataset=dataset,
        config=run_config,
        comparison_id=comparison_id,
        export_openai_trace=EXPORT_OPENAI_TRACE,
        include_payloads=INCLUDE_LOCAL_PAYLOADS,
    )

    intermediate_kinds = {"program", "function_call", "tool_output", "program_output"}
    for arm, run in trace_comparison.runs.items():
        evaluation = evaluate_inventory_run(run, dataset)
        cost = estimate_run_cost(run, pricing)
        arm_events = [event for event in trace_comparison.events if event.arm == arm]
        trace_metric_rows.append({
            "arm": arm,
            "quality_passed": evaluation.passed,
            "model_requests": len(run.requests),
            "host_round_trips": len(run.requests),
            "tool_calls": len(run.tool_calls),
            "intermediate_payload_bytes": sum(
                event.payload_bytes for event in arm_events if event.kind in intermediate_kinds
            ),
            "input_tokens": run.usage.input_tokens,
            "output_tokens": run.usage.output_tokens,
            "reasoning_tokens": run.usage.reasoning_output_tokens,
            "latency_seconds": round(run.total_latency_seconds, 3),
            "estimated_cost_usd": round(cost.total_cost, 6),
        })
        if not evaluation.passed:
            print(f"{arm} failed quality gates: {evaluation.failures}")

    display(Markdown(markdown_table(trace_metric_rows)))
    print({"comparison_id": trace_comparison.comparison_id, "trace_id": trace_comparison.trace_id})
    if trace_comparison.trace_id:
        display(Markdown("Open the [OpenAI Traces dashboard](https://platform.openai.com/traces) and search for the trace ID above."))



<h2 id="5.-Inspect-the-normalized-semantic-timeline">5. Inspect the normalized semantic timeline<a class="anchor-link" href="#5.-Inspect-the-normalized-semantic-timeline">¶</a></h2><p>The Notebook timeline is the reproducible companion to the Dashboard trace. It retains event order and comparison fields even when Trace export is disabled.</p>


In [ ]:
if trace_comparison is None:
    print("No live timeline. Enable RUN_LIVE and rerun the comparison cell.")
else:
    timeline_columns = (
        "sequence", "arm", "elapsed_ms", "duration_ms", "event",
        "name", "request", "call_id", "caller_id", "payload_bytes",
    )
    timeline_rows = trace_comparison.timeline_rows()
    for arm, title in (("direct", "Direct"), ("programmatic", "Programmatic")):
        arm_timeline_rows = [row for row in timeline_rows if row["arm"] == arm]
        arm_rows = []
        for sequence, row in enumerate(arm_timeline_rows, start=1):
            display_row = {column: row[column] for column in timeline_columns}
            display_row["sequence"] = sequence
            arm_rows.append(display_row)
        display(Markdown(f"### {title} timeline\n\n{markdown_table(arm_rows)}"))



<h2 id="6.-How-to-interpret-the-normalized-timeline">6. How to interpret the normalized timeline<a class="anchor-link" href="#6.-How-to-interpret-the-normalized-timeline">¶</a></h2><h3 id="What-the-event-order-shows">What the event order shows<a class="anchor-link" href="#What-the-event-order-shows">¶</a></h3><ul>
<li><strong>Direct:</strong> The first model request emits the inventory, demand, and inbound-shipment function calls. The host executes those calls, returns their tool outputs, and a second model request produces the final assistant message. Direct function calls have no program <code>caller_id</code>.</li>
<li><strong>Programmatic:</strong> The model first emits a <code>program</code>. Function calls made by that program share its <code>call_id</code> as their <code>caller_id</code>, which makes the parent-child relationship visible. After the program has gathered and reduced the tool results, a <code>program_output</code> is followed by the final assistant message.</li>
<li><strong>Sequence numbers:</strong> Each table starts at 1 for readability. They represent order within an arm, while <code>elapsed_ms</code> is measured from the beginning of the complete comparison, so the Programmatic table naturally starts after the Direct run.</li>
</ul>
<h3 id="What-this-saved-run-indicates">What this saved run indicates<a class="anchor-link" href="#What-this-saved-run-indicates">¶</a></h3><h4 id="Small-dataset-run">Small dataset run<a class="anchor-link" href="#Small-dataset-run">¶</a></h4><p>The interpretation below is based on the saved outputs from the <code>small</code> dataset run.</p>
<p>Reference outputs: <span class="artifact-reference" title="Standalone output artifacts are not published">live comparison trace export</span> and <span class="artifact-reference" title="Standalone output artifacts are not published">normalized semantic timeline</span>.</p>
<p>Both arms passed the quality gate and executed the same nine business-tool calls, so their resource measurements are comparable.</p>
<table>
<thead>
<tr>
<th>measure</th>
<th style="text-align:right">Direct</th>
<th style="text-align:right">Programmatic</th>
<th>observation</th>
</tr>
</thead>
<tbody>
<tr>
<td>Model requests / host round trips</td>
<td style="text-align:right">2</td>
<td style="text-align:right">5</td>
<td>Programmatic required three additional Responses API continuations in this run.</td>
</tr>
<tr>
<td>Intermediate payload bytes</td>
<td style="text-align:right">5,042</td>
<td style="text-align:right">11,304</td>
<td>The generated program and its observable intermediate events increased serialized payload volume.</td>
</tr>
<tr>
<td>Input tokens</td>
<td style="text-align:right">2,080</td>
<td style="text-align:right">3,108</td>
<td>More continuations carried more input context.</td>
</tr>
<tr>
<td>Output tokens</td>
<td style="text-align:right">385</td>
<td style="text-align:right">850</td>
<td>Program generation and intermediate control output increased generated tokens.</td>
</tr>
<tr>
<td>Reasoning tokens</td>
<td style="text-align:right">119</td>
<td style="text-align:right">265</td>
<td>Programmatic used 146 more reasoning tokens (122.7% more).</td>
</tr>
<tr>
<td>End-to-end latency</td>
<td style="text-align:right">10.715 s</td>
<td style="text-align:right">14.524 s</td>
<td>Programmatic was 3.809 seconds slower in this single run.</td>
</tr>
<tr>
<td>Estimated cost</td>
<td style="text-align:right">$0.023922</td>
<td style="text-align:right">$0.038101</td>
<td>Programmatic cost $0.014179 more for this run.</td>
</tr>
</tbody>
</table>
<p>The important conclusion is not that one orchestration style is always cheaper. In this fixed fan-out inventory case, Direct calling was more efficient because all nine independent calls could be requested together. Programmatic Tool Calling is more likely to justify its overhead when tool selection, branching, looping, or in-program aggregation avoids returning large intermediate results through repeated host-managed turns.</p>
<h4 id="Medium-dataset-run">Medium dataset run<a class="anchor-link" href="#Medium-dataset-run">¶</a></h4><p>The interpretation below is based on the saved outputs from the <code>medium</code> dataset run.</p>
<p>Reference outputs: <span class="artifact-reference" title="Standalone output artifacts are not published">live comparison trace export</span> and <span class="artifact-reference" title="Standalone output artifacts are not published">normalized semantic timeline</span>.</p>
<p>Both arms passed the quality gate and executed the same 30 business-tool calls, so their resource measurements are comparable.</p>
<table>
<thead>
<tr>
<th>measure</th>
<th style="text-align:right">Direct</th>
<th style="text-align:right">Programmatic</th>
<th>observation</th>
</tr>
</thead>
<tbody>
<tr>
<td>Model requests / host round trips</td>
<td style="text-align:right">2</td>
<td style="text-align:right">3</td>
<td>Programmatic required one additional Responses API continuation in this run.</td>
</tr>
<tr>
<td>Intermediate payload bytes</td>
<td style="text-align:right">16,805</td>
<td style="text-align:right">25,301</td>
<td>The generated program and observable intermediate events added 8,496 serialized bytes.</td>
</tr>
<tr>
<td>Input tokens</td>
<td style="text-align:right">4,578</td>
<td style="text-align:right">3,158</td>
<td>Programmatic used 1,420 fewer input tokens (31.0% less).</td>
</tr>
<tr>
<td>Output tokens</td>
<td style="text-align:right">1,061</td>
<td style="text-align:right">920</td>
<td>Programmatic used 141 fewer output tokens (13.3% less).</td>
</tr>
<tr>
<td>Reasoning tokens</td>
<td style="text-align:right">267</td>
<td style="text-align:right">267</td>
<td>Both arms used the same number of reasoning tokens.</td>
</tr>
<tr>
<td>End-to-end latency</td>
<td style="text-align:right">15.873 s</td>
<td style="text-align:right">23.616 s</td>
<td>Programmatic was 7.743 seconds slower in this single run.</td>
</tr>
<tr>
<td>Estimated cost</td>
<td style="text-align:right">$0.059780</td>
<td style="text-align:right">$0.040352</td>
<td>Programmatic cost $0.019428 less (32.5% lower).</td>
</tr>
</tbody>
</table>
<p>In this <code>medium</code> fixed-fan-out run, Programmatic Tool Calling traded latency and observable payload volume for lower model-token usage and estimated cost. The program coordinated the 30 business-tool calls and reduced their detailed results before the final model response, which more than offset the extra continuation and program-generation overhead in billed tokens. Direct calling remained faster because it requested all independent calls together and needed only one continuation.</p>
<h4 id="Why-the-cost-direction-changes-between-the-saved-runs">Why the cost direction changes between the saved runs<a class="anchor-link" href="#Why-the-cost-direction-changes-between-the-saved-runs">¶</a></h4><p>First, the direction is the opposite of what the question suggests when we look at the saved results.</p>
<table>
<thead>
<tr>
<th>Dataset</th>
<th style="text-align:right">Direct</th>
<th style="text-align:right">Programmatic</th>
<th>Result</th>
</tr>
</thead>
<tbody>
<tr>
<td>Small</td>
<td style="text-align:right">$0.023922</td>
<td style="text-align:right">$0.038101</td>
<td>Direct was $0.014179 cheaper.</td>
</tr>
<tr>
<td>Medium</td>
<td style="text-align:right">$0.059780</td>
<td style="text-align:right">$0.040352</td>
<td>Programmatic was $0.019428 cheaper.</td>
</tr>
</tbody>
</table>
<p>This cost reversal was caused less by dataset size alone than by differences in how the generated Programmatic execution handled tool calls and continuations in the two saved runs.</p>
<h5 id="Why-Direct-was-cheaper-for-the-small-dataset">Why Direct was cheaper for the small dataset<a class="anchor-link" href="#Why-Direct-was-cheaper-for-the-small-dataset">¶</a></h5><p>For the <code>small</code> run, both approaches executed the same nine tool calls, but their resource usage differed as follows.</p>
<table>
<thead>
<tr>
<th>Metric</th>
<th style="text-align:right">Direct</th>
<th style="text-align:right">Programmatic</th>
</tr>
</thead>
<tbody>
<tr>
<td>Model requests</td>
<td style="text-align:right">2</td>
<td style="text-align:right">5</td>
</tr>
<tr>
<td>Input tokens</td>
<td style="text-align:right">2,080</td>
<td style="text-align:right">3,108</td>
</tr>
<tr>
<td>Output tokens</td>
<td style="text-align:right">385</td>
<td style="text-align:right">850</td>
</tr>
<tr>
<td>Reasoning tokens</td>
<td style="text-align:right">119</td>
<td style="text-align:right">265</td>
</tr>
</tbody>
</table>
<p>The timeline shows that Programmatic made the inventory and demand calls in the first request, then handled the inbound-shipment calls one at a time across three subsequent requests, and finally generated the answer in a fifth request.</p>
<p>In other words, the following fixed overheads were large relative to the size of the task:</p>
<ul>
<li>generating the program;</li>
<li>making three additional continuations;</li>
<li>processing additional input tokens to carry forward the prior execution state; and</li>
<li>generating Programmatic control output and reasoning tokens.</li>
</ul>
<p>In the pricing snapshot used by this notebook, output tokens cost six times as much as uncached input tokens. Reasoning tokens are included in <code>output_tokens</code> and are billed at the output-token rate, so the 465 additional output tokens used by Programmatic, including 146 additional reasoning tokens, had a substantial effect on cost.</p>
<h5 id="Why-Programmatic-was-cheaper-for-the-medium-dataset">Why Programmatic was cheaper for the medium dataset<a class="anchor-link" href="#Why-Programmatic-was-cheaper-for-the-medium-dataset">¶</a></h5><p>For the <code>medium</code> run, both approaches executed 30 tool calls, but the result changed.</p>
<table>
<thead>
<tr>
<th>Metric</th>
<th style="text-align:right">Direct</th>
<th style="text-align:right">Programmatic</th>
</tr>
</thead>
<tbody>
<tr>
<td>Model requests</td>
<td style="text-align:right">2</td>
<td style="text-align:right">3</td>
</tr>
<tr>
<td>Input tokens</td>
<td style="text-align:right">4,578</td>
<td style="text-align:right">3,158</td>
</tr>
<tr>
<td>Output tokens</td>
<td style="text-align:right">1,061</td>
<td style="text-align:right">920</td>
</tr>
<tr>
<td>Reasoning tokens</td>
<td style="text-align:right">267</td>
<td style="text-align:right">267</td>
</tr>
</tbody>
</table>
<p>Direct returned the detailed results of all 30 tool calls to the next model request, where the model synthesized them into the final answer. As the dataset grew, both input and output token usage increased substantially.</p>
<p>By contrast, the Programmatic execution for the <code>medium</code> run:</p>
<ol>
<li>executed 20 inventory and demand calls in the first request;</li>
<li>executed all ten inbound-shipment calls together in the second request; and</li>
<li>generated the final answer from the aggregated result in the third request.</li>
</ol>
<p>Unlike the <code>small</code> run, where the inbound calls were processed one at a time, all ten inbound calls were grouped into a single continuation. As a result, despite the cost of generating the program, Programmatic used:</p>
<ul>
<li>1,420 fewer input tokens;</li>
<li>141 fewer output tokens; and</li>
<li>the same number of reasoning tokens.</li>
</ul>
<p>Those token savings were large enough to outweigh Programmatic Tool Calling's fixed overhead.</p>
<h5 id="How-to-interpret-the-result-accurately">How to interpret the result accurately<a class="anchor-link" href="#How-to-interpret-the-result-accurately">¶</a></h5><p>Two effects appeared together in these saved runs:</p>
<ul>
<li><strong>Scale effect:</strong> As the dataset grows, the cost of returning every tool result to the model for Direct Tool Calling and asking the model to process the full set grows quickly.</li>
<li><strong>Execution-shape effect:</strong> The saved <code>small</code> Programmatic run required five model requests, whereas the saved <code>medium</code> Programmatic run required only three. The generated programs used different batching patterns.</li>
</ul>
<p>The current results therefore do not establish that Programmatic Tool Calling will always become cheaper above a particular dataset size. The structure of the generated program and its batching behavior can vary between executions. To identify a reliable crossover point, repeat both arms with isolated cache keys and compare the mean and variance of request count, token usage, cost, latency, and quality pass rate.</p>
<p>Also, <code>intermediate_payload_bytes</code> measures the size of locally serialized observable data, not billable API tokens. This is why Programmatic can have a larger payload in the <code>medium</code> run while still producing a lower estimated cost.</p>
<p>Evidence files:</p>
<ul>
<li><span class="artifact-reference" title="Standalone output artifacts are not published">Small comparison</span></li>
<li><span class="artifact-reference" title="Standalone output artifacts are not published">Small timeline</span></li>
<li><span class="artifact-reference" title="Standalone output artifacts are not published">Medium comparison</span></li>
<li><span class="artifact-reference" title="Standalone output artifacts are not published">Medium timeline</span></li>
</ul>
<h3 id="Column-and-export-caveats">Column and export caveats<a class="anchor-link" href="#Column-and-export-caveats">¶</a></h3><ul>
<li><code>duration_ms</code> is useful primarily on <code>model_request</code> and tool-execution rows; zero or near-zero values on semantic marker rows are expected.</li>
<li><code>payload_bytes</code> measures locally serialized observable event data, not billed tokens. Use the API usage fields and pricing calculation for cost comparisons.</li>
<li><code>call_id</code> identifies an individual program or function call; <code>caller_id</code> links a Programmatic function call to the program that issued it.</li>
<li>The saved 403 <code>zdr_forbidden</code> message affects OpenAI Dashboard trace ingestion only. The local normalized timeline remains valid because it is recorded independently of trace export.</li>
</ul>



<h2 id="Exercise">Exercise<a class="anchor-link" href="#Exercise">¶</a></h2><p>Implement a quality-adjusted comparison: return the Programmatic-minus-Direct delta for a metric only when both arms pass. Try <code>host_round_trips</code>, <code>intermediate_payload_bytes</code>, and <code>estimated_cost_usd</code>.</p>


In [ ]:
def quality_adjusted_delta(rows: list[dict[str, object]], metric: str) -> float | None:
    by_arm = {str(row["arm"]): row for row in rows}
    if set(by_arm) != {"direct", "programmatic"}:
        return None
    if not all(bool(row["quality_passed"]) for row in by_arm.values()):
        return None
    return float(by_arm["programmatic"][metric]) - float(by_arm["direct"][metric])

quality_adjusted_delta(trace_metric_rows, "host_round_trips")



<h2 id="Pitfalls-and-extensions">Pitfalls and extensions<a class="anchor-link" href="#Pitfalls-and-extensions">¶</a></h2><ul>
<li><strong>Common mistake:</strong> Treating a trace as hidden chain-of-thought. The trace contains observable requests, response items, and tool execution spans only.</li>
<li><strong>Sensitive data:</strong> Full payloads are retained locally only. Set <code>INCLUDE_LOCAL_PAYLOADS=False</code> before adapting this code to real data; exported Trace spans always contain bounded summaries.</li>
<li><strong>Trace delivery:</strong> Export is buffered; the adapter flushes after the parent trace closes, but Dashboard appearance can still be delayed.</li>
<li><strong>Optional extension:</strong> Add an incident trace where each result can change the next tool decision, then compare why Direct calling may be the better orchestration shape.</li>
</ul>
<p>Official reference: <a href="https://developers.openai.com/api/docs/guides/tools-programmatic-tool-calling">Programmatic Tool Calling</a></p>
